# 04 — Agent Demo: Perceive → Reason → Act → Reflect

End-to-end demonstration of the full agentic loop on a simulated event stream.

**Covers:**
- DQN agent vs Rule-based baseline on same events
- Per-event decision logging
- Response metric comparison (F1, FPR, MTTR, cumulative reward)
- RL training reward curves

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import yaml

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('../configs/train_config.yaml') as f:
    cfg = yaml.safe_load(f)
with open('../configs/agent_config.yaml') as f:
    agent_cfg = yaml.safe_load(f)

full_cfg = {**cfg, **agent_cfg}
full_cfg['class_names'] = cfg['dataset']['class_names']
print('Configs loaded')

In [ ]:
# ── DQN vs Rule-Based Comparison ──────────────────────────────────────
agent_comparison_path = '../results/agent_comparison.json'

if os.path.exists(agent_comparison_path):
    with open(agent_comparison_path) as f:
        results = json.load(f)

    rows = []
    for policy, res in results.items():
        if 'status' not in res:
            rows.append({'Policy': policy, **res})

    df = pd.DataFrame(rows).set_index('Policy')
    print('=== AGENT POLICY COMPARISON ===')
    print(df[['f1', 'fpr', 'precision', 'recall', 'mean_response_time_ms', 'total_reward']].to_string())

    # Visual comparison
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    for ax, (metric, title, higher_better) in zip(axes, [
        ('f1',  'F1 Score', True),
        ('fpr', 'False Positive Rate', False),
        ('mean_response_time_ms', 'MTTR (ms)', False),
    ]):
        if metric in df.columns:
            vals = df[metric]
            best_idx = vals.idxmax() if higher_better else vals.idxmin()
            colors = ['#1D9E75' if i == best_idx else '#185FA5' for i in vals.index]
            vals.plot(kind='bar', ax=ax, color=colors, alpha=0.85)
            ax.set_title(title)
            ax.set_ylabel(metric)
            if metric in ('f1', 'fpr'):
                ax.set_ylim([0, 1])
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=0)

    plt.suptitle('Agent Policy Comparison (green = best)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../results/figures/agent_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[INFO] Agent comparison not yet available.')
    print('Run:')
    print('  python run_agent.py --mode train_rl')
    print('  python scripts/evaluate.py')

In [ ]:
# ── RL Training Reward Curves ─────────────────────────────────────────
rl_history_path = '../results/checkpoints/rl_history.json'

if os.path.exists(rl_history_path):
    with open(rl_history_path) as f:
        rl_history = json.load(f)

    rewards = rl_history['episode_rewards']
    # Rolling average
    window = min(10, len(rewards))
    rolling_avg = pd.Series(rewards).rolling(window).mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(rewards, alpha=0.3, color='#185FA5', label='Episode reward')
    ax.plot(rolling_avg, color='#E24B4A', linewidth=2, label=f'{window}-episode rolling avg')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Cumulative Reward')
    ax.set_title('DQN Agent Training — Reward Curve')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../results/figures/rl_reward_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Final episode reward  : {rewards[-1]:.2f}')
    print(f'Best episode reward   : {max(rewards):.2f}')
    print(f'Training convergence  : episode {np.argmax(rolling_avg.dropna().values)}')
else:
    print('[INFO] RL training history not yet available.')
    print('Run: python run_agent.py --mode train_rl')

In [ ]:
# ── Sample Incident Log Inspection ────────────────────────────────────
import glob
log_files = glob.glob('../results/incident_logs/*.jsonl')

if log_files:
    records = []
    for lf in log_files:
        with open(lf) as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

    df_incidents = pd.DataFrame(records)
    print(f'Total logged decisions: {len(df_incidents)}')

    if len(df_incidents) > 0:
        print('\nAction distribution:')
        print(df_incidents['action'].value_counts())

        print('\nTop threat types detected:')
        print(df_incidents['predicted_threat'].value_counts().head(5))

        print('\nSample decision with reasoning:')
        non_ignore = df_incidents[df_incidents['action'] != 'IGNORE']
        if len(non_ignore) > 0:
            sample = non_ignore.iloc[0]
            print(f"  Threat    : {sample['predicted_threat']}")
            print(f"  Action    : {sample['action']}")
            print(f"  Confidence: {sample['confidence']}")
            print(f"  Reasoning : {sample['reasoning']}")
else:
    print('[INFO] No incident logs yet.')
    print('Run: python run_agent.py --mode simulate')